In [1]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster(n_workers=4, threads_per_worker=1,  memory_limit='6GB')
client = Client(cluster)

In [2]:
from tropical_cyclones import TCs, multi_plot, plot_trajectories
from aqua.core.util import load_yaml

config = load_yaml('/home/mccorda/work/tc_analysis3/config_tcs_yearlystitch.yaml')

/work/users/mccorda/miniforge3/envs/aqua-diagnostics/lib/python3.12/site-packages/intake_esm/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [ ]:
# IBTrACS → Custom Format + Plotting

#This notebook:
#1. Converts IBTrACS NetCDF to the 12-column ASCII format expected by `getTrajectories_direct()`
#2. Loads and plots the converted tracks using your existing `plotting_TCs_custom.py`
#3. Shows the difference between IBTrACS Saffir-Simpson (wind-based) and your SLP-based categories


#**Pressure units reminder:**
#- IBTrACS stores pressure in **mb (hPa)**
#- ERA5 / TempestExtremes uses **Pa**
#- The converter multiplies by 100: `mb × 100 → Pa`
#- Your reader divides by 100: `Pa / 100 → hPa` 

#**Saffir-Simpson in IBTrACS (`usa_sshs`):** wind-based (kts)  
#**Saffir-Simpson in your code (`category_from_slp_pa`):** SLP-based (Pa) — **different definition!**

In [3]:
import sys, os

# ── EDIT THESE ──────────────────────────────────────────────────────────────

# Path to IBTrACS NetCDF file
IBTRACS_NC   = '/work/users/jost/IBTrACS.ALL.v04r01.nc'

# Where to write the converted ASCII file
CONVERTED_TXT = '/home/mccorda/work/tc_analysis_ibtracks/ibtracs_1979_2014_custom.txt'

# Where your plotting_TCs_custom.py lives
PLOTTING_PY_DIR = '/home/mccorda/work/AQUA-diagnostics/frontier-diagnostics/tropical_cyclones/tropical_cyclones/plots/plotting_TCs_custom_memory.py/'

# Where the ibtracs_to_custom_format.py converter lives
CONVERTER_DIR = '/home/mccorda/work/AQUA-diagnostics/frontier-diagnostics/tropical_cyclones/tropical_cyclones/plots/ibtracs_to_custom_format.py/'

# Output directory for plots
PLOTDIR = '/home/mccorda/work/tc_analysis_ibtracks/Figures/'

START_YEAR = 1979
END_YEAR   = 2014

# ── add scripts to path ──────────────────────────────────────────────────────
for d in [PLOTTING_PY_DIR, CONVERTER_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)

print('Paths configured.')

Paths configured.


In [ ]:
from tropical_cyclones import convert

# Skip if already converted
if os.path.exists(CONVERTED_TXT):
    print(f'File already exists: {CONVERTED_TXT}')
    print('Delete it to re-convert, or continue to the next cell.')
else:
    convert(IBTRACS_NC, CONVERTED_TXT, START_YEAR, END_YEAR)


IBTrACS → custom format converter
  Input  : /work/users/jost/IBTrACS.ALL.v04r01.nc
  Output : /home/mccorda/work/tc_analysis_ibtracks/ibtracs_1979_2014_custom.txt
  Period : 1979–2014

Dataset: 13553 storms × 360 time steps
Loading variables …
  Wind source : wmo_wind
  Pres source : wmo_pres
  Processing storm 0/13553 …
  Processing storm 2000/13553 …


In [ ]:
#check on the converted file
import numpy as np

# Count lines, print first 5 data rows
with open(CONVERTED_TXT) as f:
    lines = [l for l in f if not l.startswith('#')]

print(f'Total data lines : {len(lines):,}')
print(f'\nFirst 5 rows:')
for l in lines[:5]:
    print(' ', l.rstrip())

# Check column count
col_counts = [len(l.split()) for l in lines]
print(f'\nColumn counts (should all be 12): {set(col_counts)}')

In [ ]:
# load trajectories with my reader
from tropical_cyclones.plots.plotting_TCs_custom_memory import getTrajectories_direct

nstorms, max_pts, trajectories = getTrajectories_direct(CONVERTED_TXT)

print(f'\nStorms loaded : {nstorms}')
print(f'Max timesteps : {max_pts}')

# Show one example storm
example_id = list(trajectories.keys())[0]
ex = trajectories[example_id]
print(f'\nExample storm : {example_id}')
print(f'  Timesteps   : {len(ex["lon"])}')
print(f'  Lon range   : {ex["lon"].min():.1f} – {ex["lon"].max():.1f}')
print(f'  Lat range   : {ex["lat"].min():.1f} – {ex["lat"].max():.1f}')
print(f'  SLP (hPa)   : {ex["slp"].min()/100:.1f} – {ex["slp"].max()/100:.1f}  [stored as Pa, shown here as hPa]')
print(f'  Wind (kts)  : {ex["wind"].min():.0f} – {ex["wind"].max():.0f}')

In [ ]:
#build tdict
tdict = {
    'dataset': {
        'model': 'IBTrACS',
        'exp'  : 'v04r01_OBS'
    },
    'time': {
        'startdate': f'{START_YEAR}-01-01',
        'enddate'  : f'{END_YEAR}-12-31'
    },
    'paths': {
        'plotdir': PLOTDIR
    }
}
print('tdict ready:', tdict)

In [ ]:
from plotting_TCs_custom import plot_trajectories_direct
plot_trajectories_direct(CONVERTED_TXT, tdict)

In [ ]:
from plotting_TCs_custom import plot_trajectories_colored
plot_trajectories_colored(CONVERTED_TXT, tdict, color_by='category')

In [ ]:
from plotting_TCs_custom import plot_track_density_grid
plot_track_density_grid(CONVERTED_TXT, tdict, grid_size=2.5)

In [ ]:
# Comparison: IBTrACS usa_sshs (wind-based) vs your SLP-based category

#This cell reads `usa_sshs` directly from the NetCDF and compares it
#with `category_from_slp_pa()` applied to `wmo_pres`.
#**Expected result:** the two classifications differ, especially near category boundaries,
#because the wind–pressure relationship is not fixed (it varies by basin and storm size).

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from plotting_TCs_custom import category_from_slp_pa

ds = nc.Dataset(IBTRACS_NC, 'r')

sshs_all  = ds.variables['usa_sshs'][:].filled(-15)   # wind-based, -15 = missing
pres_all  = ds.variables['wmo_pres'][:].filled(-9999) # mb
iso_raw   = ds.variables['iso_time'][:]               # to filter years

cat_wind_list = []
cat_slp_list  = []

nstorms = sshs_all.shape[0]

for i in range(nstorms):
    for j in range(sshs_all.shape[1]):
        # year filter via iso_time
        s = b''.join(iso_raw[i, j].filled(b' ')).decode('utf-8', errors='replace').strip()
        if not s or len(s) < 4:
            continue
        try:
            yr = int(s[:4])
        except ValueError:
            continue
        if not (START_YEAR <= yr <= END_YEAR):
            continue

        sshs_val = int(sshs_all[i, j])
        pres_val = float(pres_all[i, j])

        if sshs_val == -15 or pres_val <= 0 or pres_val == -9999:
            continue

        # Only compare where both are valid and storm is tropical (sshs >= -1)
        if sshs_val < -1:
            continue

        cat_slp = category_from_slp_pa(pres_val * 100.0)  # mb → Pa

        # Remap sshs to 0–5 (clip -1 TD to 0)
        cat_wind = max(0, sshs_val)

        cat_wind_list.append(cat_wind)
        cat_slp_list.append(cat_slp)

ds.close()

cat_wind_arr = np.array(cat_wind_list)
cat_slp_arr  = np.array(cat_slp_list)

print(f'Valid paired observations: {len(cat_wind_arr):,}')

# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
labels = [0, 1, 2, 3, 4, 5]
cm = confusion_matrix(cat_wind_arr, cat_slp_arr, labels=labels)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['TD/TS','Cat1','Cat2','Cat3','Cat4','Cat5'])
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_xlabel('SLP-based category (your definition)', fontsize=12)
ax.set_ylabel('Wind-based category (IBTrACS usa_sshs)', fontsize=12)
ax.set_title(f'Category agreement: wind vs SLP\nIBTrACS {START_YEAR}–{END_YEAR}', fontsize=13)
plt.tight_layout()

os.makedirs(PLOTDIR, exist_ok=True)
plt.savefig(os.path.join(PLOTDIR, f'category_comparison_wind_vs_slp_{START_YEAR}_{END_YEAR}.pdf'),
            dpi=200, bbox_inches='tight')
plt.show()

# Agreement percentage
agree = np.sum(cat_wind_arr == cat_slp_arr)
print(f'\nExact agreement : {agree:,} / {len(cat_wind_arr):,} = {100*agree/len(cat_wind_arr):.1f}%')